# TEXT CLUSTERING USING TF-IDF VECTORIZER

In [10]:
import numpy as np 
from sklearn.cluster import KMeans 
from sklearn.feature_extraction.text import TfidfVectorizer 
from tabulate import tabulate 
from collections import Counter

In [11]:
dataset = ["I love playing football on the weekends", 
           "I enjoy hiking and camping in the mountains", 
           "I like to read books and watch movies", 
           "I prefer playing video games over sports", 
           "I love listening to music and going to concerts"]

In [12]:
vectorizer = TfidfVectorizer() 
X = vectorizer.fit_transform(dataset)

In [13]:
k = 2  # Define the number of clusters 
km = KMeans(n_clusters=k) 
km.fit(X) 
 
# Predict the clusters for each document 
y_pred = km.predict(X) 
 
# Display the document and its predicted cluster in a table 
table_data = [["Document", "Predicted Cluster"]] 
table_data.extend([[doc, cluster] for doc, cluster in zip(dataset, y_pred)]) 
print(tabulate(table_data, headers="firstrow"))
# Print top terms per cluster 
print("\nTop terms per cluster:") 
order_centroids = km.cluster_centers_.argsort()[:, ::-1] 
terms = vectorizer.get_feature_names_out() 
for i in range(k): 
    print("Cluster %d:" % i) 
    for ind in order_centroids[i, :10]: 
        print(' %s' % terms[ind]) 
    print()

Document                                           Predicted Cluster
-----------------------------------------------  -------------------
I love playing football on the weekends                            1
I enjoy hiking and camping in the mountains                        0
I like to read books and watch movies                              0
I prefer playing video games over sports                           1
I love listening to music and going to concerts                    0

Top terms per cluster:
Cluster 0:
 to
 and
 read
 like
 enjoy
 camping
 hiking
 mountains
 in
 movies

Cluster 1:
 playing
 on
 weekends
 football
 over
 sports
 video
 games
 prefer
 the



In [14]:
# Calculate purity 
total_samples = len(y_pred) 
cluster_label_counts = [Counter(y_pred)] 
purity = sum(max(cluster.values()) for cluster in cluster_label_counts) / total_samples 
print("Purity:", purity)

Purity: 0.6


# TEXT CLUSTERING USING WORD2VEC VECTORIZER

In [23]:
pip install gensim

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [28]:
import numpy as np 
from sklearn.cluster import KMeans 
from gensim.models import Word2Vec 
from tabulate import tabulate 
from collections import Counter

In [29]:
dataset = ["I love playing football on the weekends", 
           "I enjoy hiking and camping in the mountains", 
           "I like to read books and watch movies", 
           "I prefer playing video games over sports", 
           "I love listening to music and going to concerts"]

In [31]:
tokenized_dataset = [doc.split() for doc in dataset] 
word2vec_model = Word2Vec(sentences=tokenized_dataset, vector_size=100, 
window=5, min_count=1, workers=4)

In [32]:
X = np.array([np.mean([word2vec_model.wv[word] for word in doc.split() if word in 
word2vec_model.wv], axis=0) for doc in dataset])

In [33]:
k = 2  # Define the number of clusters 
km = KMeans(n_clusters=k) 
km.fit(X) 
 
# Predict the clusters for each document 
y_pred = km.predict(X) 
 
# Tabulate the document and predicted cluster 
table_data = [["Document", "Predicted Cluster"]] 
table_data.extend([[doc, cluster] for doc, cluster in zip(dataset, y_pred)]) 
print(tabulate(table_data, headers="firstrow"))

Document                                           Predicted Cluster
-----------------------------------------------  -------------------
I love playing football on the weekends                            0
I enjoy hiking and camping in the mountains                        1
I like to read books and watch movies                              1
I prefer playing video games over sports                           1
I love listening to music and going to concerts                    0


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


In [34]:
# Calculate purity 
total_samples = len(y_pred) 
cluster_label_counts = [Counter(y_pred)] 
purity = sum(max(cluster.values()) for cluster in cluster_label_counts) / total_samples 
print("Purity:", purity) 

Purity: 0.6


# EXERCISE 1

In [35]:
import re
import numpy as np
import nltk
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec
from tabulate import tabulate
from collections import Counter
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [36]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [37]:
dataset = [
    "I love playing football on the weekends",
    "I enjoy hiking and camping in the mountains",
    "I like to read books and watch movies",
    "I prefer playing video games over sports",
    "I love listening to music and going to concerts"
]

In [38]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()                         # lowercase
    text = re.sub(r'[^a-z\s]', '', text)       # remove punctuation/numbers
    words = text.split()                       # tokenize
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return " ".join(words)

In [39]:
vectorizer_raw = TfidfVectorizer()
X_raw = vectorizer_raw.fit_transform(dataset)

k = 2
km_raw = KMeans(n_clusters=k, random_state=42, n_init=10)
km_raw.fit(X_raw)

y_pred_raw = km_raw.predict(X_raw)

table_data = [["Document", "Predicted Cluster"]]
table_data.extend([[doc, cluster] for doc, cluster in zip(dataset, y_pred_raw)])
print(tabulate(table_data, headers="firstrow"))

print("\nTop terms per cluster:")
order_centroids = km_raw.cluster_centers_.argsort()[:, ::-1]
terms = vectorizer_raw.get_feature_names_out()

for i in range(k):
    print(f"Cluster {i}:")
    for ind in order_centroids[i, :10]:
        print(terms[ind])
    print()

total_samples = len(y_pred_raw)
cluster_label_counts = [Counter(y_pred_raw)]
purity_raw_tfidf = sum(max(cluster.values()) for cluster in cluster_label_counts) / total_samples
print("Purity without preprocessing (TF-IDF):", purity_raw_tfidf)

Document                                           Predicted Cluster
-----------------------------------------------  -------------------
I love playing football on the weekends                            0
I enjoy hiking and camping in the mountains                        0
I like to read books and watch movies                              1
I prefer playing video games over sports                           0
I love listening to music and going to concerts                    1

Top terms per cluster:
Cluster 0:
playing
the
weekends
on
football
prefer
video
over
sports
games

Cluster 1:
to
and
read
like
books
movies
watch
music
listening
going

Purity without preprocessing (TF-IDF): 0.6


In [40]:
dataset_clean = [preprocess_text(doc) for doc in dataset]

print("Preprocessed dataset:")
for doc in dataset_clean:
    print(doc)

Preprocessed dataset:
love playing football weekend
enjoy hiking camping mountain
like read book watch movie
prefer playing video game sport
love listening music going concert


In [41]:
vectorizer_clean = TfidfVectorizer()
X_clean = vectorizer_clean.fit_transform(dataset_clean)

km_clean = KMeans(n_clusters=k, random_state=42, n_init=10)
km_clean.fit(X_clean)

y_pred_clean = km_clean.predict(X_clean)

table_data = [["Document", "Predicted Cluster"]]
table_data.extend([[doc, cluster] for doc, cluster in zip(dataset_clean, y_pred_clean)])
print(tabulate(table_data, headers="firstrow"))

print("\nTop terms per cluster after preprocessing:")
order_centroids = km_clean.cluster_centers_.argsort()[:, ::-1]
terms = vectorizer_clean.get_feature_names_out()

for i in range(k):
    print(f"Cluster {i}:")
    for ind in order_centroids[i, :10]:
        print(terms[ind])
    print()

total_samples = len(y_pred_clean)
cluster_label_counts = [Counter(y_pred_clean)]
purity_clean_tfidf = sum(max(cluster.values()) for cluster in cluster_label_counts) / total_samples
print("Purity with preprocessing (TF-IDF):", purity_clean_tfidf)

Document                              Predicted Cluster
----------------------------------  -------------------
love playing football weekend                         1
enjoy hiking camping mountain                         0
like read book watch movie                            0
prefer playing video game sport                       0
love listening music going concert                    1

Top terms per cluster after preprocessing:
Cluster 0:
mountain
hiking
enjoy
camping
video
game
prefer
sport
watch
book

Cluster 1:
love
football
weekend
music
listening
going
concert
playing
watch
read

Purity with preprocessing (TF-IDF): 0.6


In [42]:
tokenized_dataset_raw = [doc.split() for doc in dataset]

word2vec_model_raw = Word2Vec(
    sentences=tokenized_dataset_raw,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

X_w2v_raw = np.array([
    np.mean([word2vec_model_raw.wv[word] for word in doc.split() if word in word2vec_model_raw.wv], axis=0)
    for doc in dataset
])

km_w2v_raw = KMeans(n_clusters=k, random_state=42, n_init=10)
km_w2v_raw.fit(X_w2v_raw)

y_pred_w2v_raw = km_w2v_raw.predict(X_w2v_raw)

table_data = [["Document", "Predicted Cluster"]]
table_data.extend([[doc, cluster] for doc, cluster in zip(dataset, y_pred_w2v_raw)])
print(tabulate(table_data, headers="firstrow"))

total_samples = len(y_pred_w2v_raw)
cluster_label_counts = [Counter(y_pred_w2v_raw)]
purity_raw_w2v = sum(max(cluster.values()) for cluster in cluster_label_counts) / total_samples
print("Purity without preprocessing (Word2Vec):", purity_raw_w2v)

Document                                           Predicted Cluster
-----------------------------------------------  -------------------
I love playing football on the weekends                            1
I enjoy hiking and camping in the mountains                        1
I like to read books and watch movies                              0
I prefer playing video games over sports                           1
I love listening to music and going to concerts                    0
Purity without preprocessing (Word2Vec): 0.6


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


In [44]:
tokenized_dataset_clean = [doc.split() for doc in dataset_clean]

word2vec_model_clean = Word2Vec(
    sentences=tokenized_dataset_clean,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

X_w2v_clean = np.array([
    np.mean([word2vec_model_clean.wv[word] for word in doc.split() if word in word2vec_model_clean.wv], axis=0)
    for doc in dataset_clean
])

km_w2v_clean = KMeans(n_clusters=k, random_state=42, n_init=10)
km_w2v_clean.fit(X_w2v_clean)

y_pred_w2v_clean = km_w2v_clean.predict(X_w2v_clean)

table_data = [["Document", "Predicted Cluster"]]
table_data.extend([[doc, cluster] for doc, cluster in zip(dataset_clean, y_pred_w2v_clean)])
print(tabulate(table_data, headers="firstrow"))

total_samples = len(y_pred_w2v_clean)
cluster_label_counts = [Counter(y_pred_w2v_clean)]
purity_clean_w2v = sum(max(cluster.values()) for cluster in cluster_label_counts) / total_samples
print("Purity with preprocessing (Word2Vec):", purity_clean_w2v)

Document                              Predicted Cluster
----------------------------------  -------------------
love playing football weekend                         0
enjoy hiking camping mountain                         0
like read book watch movie                            0
prefer playing video game sport                       0
love listening music going concert                    1
Purity with preprocessing (Word2Vec): 0.8


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


In [45]:
print("=== Purity Comparison ===")
print("TF-IDF without preprocessing :", purity_raw_tfidf)
print("TF-IDF with preprocessing    :", purity_clean_tfidf)
print("Word2Vec without preprocessing :", purity_raw_w2v)
print("Word2Vec with preprocessing    :", purity_clean_w2v)

=== Purity Comparison ===
TF-IDF without preprocessing : 0.6
TF-IDF with preprocessing    : 0.6
Word2Vec without preprocessing : 0.6
Word2Vec with preprocessing    : 0.8



> Yes, the purity differs after applying text preprocessing. This is because preprocessing removes noise such as stopwords and irrelevant terms, which helps improve the clustering quality and leads to more accurate grouping of similar documents.


# EXERCISE 2

In [46]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.cluster import KMeans

In [47]:
df = pd.read_csv("customer_complaints_1.csv")
print(df.head())
print(df.columns)

                        author      posted_on  rating  \
0  Alantae of Chesterfeild, MI  Nov. 22, 2016       1   
1     Vera of Philadelphia, PA  Nov. 19, 2016       1   
2  Sarah of Rancho Cordova, CA  Nov. 17, 2016       1   
3     Dennis of Manchester, NH  Nov. 16, 2016       1   
4         Ryan of Bellevue, WA  Nov. 14, 2016       1   

                                                text  
0  I used to love Comcast. Until all these consta...  
1  I'm so over Comcast! The worst internet provid...  
2  If I could give them a negative star or no sta...  
3  I've had the worst experiences so far since in...  
4  Check your contract when you sign up for Comca...  
Index(['author', 'posted_on', 'rating', 'text'], dtype='object')


In [49]:
def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)   # remove punctuation and numbers
    words = text.split()
    words = [word for word in words if word not in ENGLISH_STOP_WORDS and len(word) > 2]
    return " ".join(words)

In [50]:
df['clean_text'] = df['text'].apply(preprocess)
print(df[['text', 'clean_text']].head())

                                                text  \
0  I used to love Comcast. Until all these consta...   
1  I'm so over Comcast! The worst internet provid...   
2  If I could give them a negative star or no sta...   
3  I've had the worst experiences so far since in...   
4  Check your contract when you sign up for Comca...   

                                          clean_text  
0  used love comcast constant updates internet ca...  
1  comcast worst internet provider taking online ...  
2  negative star stars review worked industry bad...  
3  worst experiences far install problems shows s...  
4  check contract sign comcast advertised offers ...  


In [51]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['clean_text'])

In [52]:
k = 3
km = KMeans(n_clusters=k, random_state=42, n_init=10)
km.fit(X)

df['Cluster'] = km.labels_

In [53]:
print(df[['text', 'Cluster']])

                                                 text  Cluster
0   I used to love Comcast. Until all these consta...        1
1   I'm so over Comcast! The worst internet provid...        0
2   If I could give them a negative star or no sta...        0
3   I've had the worst experiences so far since in...        0
4   Check your contract when you sign up for Comca...        1
5   Thank God. I am changing to Dish. They gave me...        0
6   I Have been a long time customer and only have...        1
7   There is a malfunction on the DVR manager whic...        0
8   Charges overwhelming. Comcast service rep was ...        2
9   I have had cable, DISH, and U-verse, etc. in t...        0
10  Had them from 2014 to now. I'd tell new custom...        1
11  Disappointed. I have been a Comcast/Xfinity cu...        1
12  These people are unethical and disturbing obli...        1
13  Unplanned, unexpected, all day outages, rude s...        2
14  BE WARNED. You will have 10$ hidden fees when ...  

In [54]:
terms = vectorizer.get_feature_names_out()
order_centroids = km.cluster_centers_.argsort()[:, ::-1]

for i in range(k):
    print(f"\nCluster {i}:")
    for ind in order_centroids[i, :10]:
        print(terms[ind])


Cluster 0:
comcast
internet
service
customer
security
months
services
adding
finally
time

Cluster 1:
contract
mbps
speed
internet
xfinity
told
service
don
comcast
customer

Cluster 2:
rude
service
day
rep
joke
local
people
pass
overwhelming
tom
